# Tiền xử lý dữ liệu cổ phiếu Apple cho Machine Learning

Các bước:
1. Đọc dữ liệu và chuyển các cột về dạng số
2. Tách ngày / tháng / năm
3. Tạo feature: `Return_1`, `High_Low_Range`, `Open_Close_Change`
4. Tạo `Target` = `Price` của ngày t+1
5. Lưu file sạch để train mô hình

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np

INPUT_PATH = '/content/drive/MyDrive/IS403.Apple_Stock_Analysis/Raw data/Apple Stock Price History.csv'
OUTPUT_PATH = 'Apple_Stock_ready.csv'

## 1. Đọc dữ liệu

In [5]:
df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
print(df.shape)
print(df.dtypes)
df.head()

(4022, 7)
Date         object
Price       float64
Open        float64
High        float64
Low         float64
Vol.         object
Change %     object
dtype: object


,Date,Price,Open,High,Low,Vol.,Change %
0,09/29/2026,331.67,336.18,337.00,330.56,18.81M,-1.99%
1,09/28/2026,338.40,340.37,342.99,338.04,32.82M,-0.78%
2,09/25/2026,341.07,336.04,341.67,334.53,30.00M,1.53%
3,09/24/2026,335.92,336.72,338.91,334.30,24.73M,-0.33%
4,09/23/2026,337.02,341.08,341.80,335.50,31.66M,-0.80%


## 2. Chuyển về dạng số
- `Price, Open, High, Low`: bỏ dấu `,` (nếu có) rồi ép sang float
- `Vol.`: đổi hậu tố K / M / B thành số thật (`18.81M` → 18,810,000)
- `Change %`: bỏ ký hiệu `%` rồi chia 100 (`-1.99%` → -0.0199)
- `Date`: chuyển sang datetime

In [6]:
def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(',', '', regex=False), errors='coerce')

def parse_volume(s):
    s = s.astype(str).str.strip().str.replace(',', '', regex=False)
    mult = s.str[-1].map({'K': 1e3, 'M': 1e6, 'B': 1e9}).fillna(1)
    num = pd.to_numeric(s.str.rstrip('KMB'), errors='coerce')
    return num * mult

df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
for c in ['Price', 'Open', 'High', 'Low']:
    df[c] = to_num(df[c])

df['Volume'] = parse_volume(df['Vol.'])
df['Change_Pct'] = pd.to_numeric(df['Change %'].astype(str).str.replace('%', '', regex=False), errors='coerce') / 100
df = df.drop(columns=['Vol.', 'Change %'])

print(df.isna().sum())
df.dtypes

Date          0
Price         0
Open          0
High          0
Low           0
Volume        0
Change_Pct    0
dtype: int64


,0
Date,datetime64[ns]
Price,float64
Open,float64
High,float64
Low,float64
Volume,float64
Change_Pct,float64


## 3. Sắp xếp theo thời gian (cũ → mới)
Dữ liệu gốc xếp từ mới đến cũ. Cần đảo lại để `shift` và `pct_change` đúng chiều thời gian.

In [7]:
df = df.sort_values('Date').reset_index(drop=True)
print('Ngày trùng:', df['Date'].duplicated().sum())
df[['Date']].agg(['min', 'max'])

Ngày trùng: 0


,Date
min,2010-10-01
max,2026-09-29


## 4. Tách ngày tháng năm

In [8]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0 = Thứ Hai, 4 = Thứ Sáu
df[['Date', 'Year', 'Month', 'Day', 'DayOfWeek']].head()

,Date,Year,Month,Day,DayOfWeek
0,2010-10-01,2010,10,1,4
1,2010-10-04,2010,10,4,0
2,2010-10-05,2010,10,5,1
3,2010-10-06,2010,10,6,2
4,2010-10-07,2010,10,7,3


## 5. Tạo feature và target
- `Return_1` = (Price_t / Price_{t-1}) − 1
- `High_Low_Range` = High − Low
- `Open_Close_Change` = Price − Open (coi `Price` là giá đóng cửa)
- `NextPrice` = Price của ngày **t+1**

In [9]:
df['Return_1'] = df['Price'].pct_change(1)
df['High_Low_Range'] = df['High'] - df['Low']
df['Open_Close_Change'] = df['Price'] - df['Open']
df['NextPrice'] = df['Price'].shift(-1)

df[['Date', 'Price', 'Return_1', 'High_Low_Range', 'Open_Close_Change', 'NextPrice']].head()

,Date,Price,Return_1,High_Low_Range,Open_Close_Change,NextPrice
0,2010-10-01,10.09,NaN,0.18,-0.13,9.95
1,2010-10-04,9.95,-0.013875,0.18,-0.11,10.32
2,2010-10-05,10.32,0.037186,0.27,0.25,10.33
3,2010-10-06,10.33,0.000969,0.24,-0.01,10.33
4,2010-10-07,10.33,0.000000,0.12,-0.04,10.50


Kiểm tra nhanh: `Return_1` phải gần khớp với `Change_Pct` của file gốc (chỉ lệch do làm tròn), và `NextPrice` của dòng t phải bằng `Price` của dòng t+1.

In [10]:
print('Max |Return_1 - Change_Pct|:', (df['Return_1'] - df['Change_Pct']).abs().max())
print('NextPrice khớp Price ngày sau:', (df['NextPrice'].iloc[:-1].values == df['Price'].iloc[1:].values).all())

Max |Return_1 - Change_Pct|: 0.0008520871143375451
NextPrice khớp Price ngày sau: True


## 6. Xử lý NaN và lưu file
Dòng đầu không có `Return_1`, dòng cuối không có `Target` (chưa biết giá ngày hôm sau) nên bỏ 2 dòng này.

In [11]:
cols = ['Date', 'Year', 'Month', 'Day', 'DayOfWeek',
        'Price', 'Open', 'High', 'Low', 'Volume', 'Change_Pct',
        'Return_1', 'High_Low_Range', 'Open_Close_Change', 'NextPrice']

data = df.dropna().reset_index(drop=True)[cols]
data.to_csv(OUTPUT_PATH, index=False)

print(data.shape)
print('NaN còn lại:', data.isna().sum().sum())
data.head()

(4020, 15)
NaN còn lại: 0


,Date,Year,Month,Day,DayOfWeek,Price,Open,High,Low,Volume,Change_Pct,Return_1,High_Low_Range,Open_Close_Change,NextPrice
0,2010-10-04,2010,10,4,0,9.95,10.06,10.10,9.92,435300000.0,-0.0137,-0.013875,0.18,-0.11,10.32
1,2010-10-05,2010,10,5,1,10.32,10.07,10.34,10.07,501970000.0,0.0370,0.037186,0.27,0.25,10.33
2,2010-10-06,2010,10,6,2,10.33,10.34,10.43,10.19,670870000.0,0.0009,0.000969,0.24,-0.01,10.33
3,2010-10-07,2010,10,7,3,10.33,10.37,10.37,10.25,408400000.0,0.0000,0.000000,0.12,-0.04,10.50
4,2010-10-08,2010,10,8,4,10.50,10.42,10.52,10.36,658400000.0,0.0168,0.016457,0.16,0.08,10.55


## 7. Lưu file vào thư mục Cleaned data trên Drive

In [12]:
import os

# Xác định đường dẫn thư mục đích trên Drive
CLEANED_DIR = '/content/drive/MyDrive/IS403.Apple_Stock_Analysis/Cleaned data'

# Tạo thư mục nếu chưa tồn tại
os.makedirs(CLEANED_DIR, exist_ok=True)

# Đường dẫn file lưu trữ
CLEANED_FILE_PATH = os.path.join(CLEANED_DIR, 'Apple_Stock_ready.csv')

# Lưu file
data.to_csv(CLEANED_FILE_PATH, index=False)
print(f'Đã lưu file thành công tại: {CLEANED_FILE_PATH}')

Đã lưu file thành công tại: /content/drive/MyDrive/IS403.Apple_Stock_Analysis/Cleaned data/Apple_Stock_ready.csv
